# stock Qwen3 pipeline 

## 1. Install / upgrade dependencies

In [ ]:
!pip install -q -U "transformers>=4.51.0" accelerate bitsandbytes

## 2. Mount Drive and check GPU

In [ ]:
import torch

print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("No GPU detected -- check Runtime > Change runtime type > T4 GPU (or better)")

from google.colab import drive
drive.mount("/content/drive")

import os
DRIVE_PROJECT_DIR = "/content/drive/MyDrive/MSc_Diss"
os.makedirs(f"{DRIVE_PROJECT_DIR}/data", exist_ok=True)
os.makedirs(f"{DRIVE_PROJECT_DIR}/results", exist_ok=True)
print(f"Project directory on Drive: {DRIVE_PROJECT_DIR}")

## 3. Write the pipeline code

Embeds `qwen3_inference.py`, `applicant_profile.py`, `stock_pipeline.py`,
and `run_llm_pipeline.py`

In [ ]:
%%writefile qwen3_inference.py
"""
qwen3_inference.py
"""

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_NAME = "Qwen/Qwen3-8B"

# Qwen's documented recommendation for non-thinking mode.
DEFAULT_TEMPERATURE = 0.7
DEFAULT_TOP_P = 0.8
DEFAULT_TOP_K = 20


def load_model_and_tokenizer(
    model_name: str = MODEL_NAME,
    quantize_4bit: bool = True,
    compute_dtype: torch.dtype = torch.float16,
):
    """
    Loads Qwen3-8B. Requires transformers>=4.51.0 (older versions error on
    Qwen3's architecture). Requires bitsandbytes if quantize_4bit=True.
    """
    tokenizer = AutoTokenizer.from_pretrained(model_name)

    if quantize_4bit:
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=compute_dtype,
            bnb_4bit_use_double_quant=True,
        )
        model = AutoModelForCausalLM.from_pretrained(
            model_name, quantization_config=bnb_config, device_map="auto"
        )
    else:
        model = AutoModelForCausalLM.from_pretrained(
            model_name, torch_dtype="auto", device_map="auto"
        )

    return model, tokenizer


def generate(
    model,
    tokenizer,
    prompt: str,
    seed: int,
    system_prompt: str | None = None,
    max_new_tokens: int = 512,
    temperature: float = DEFAULT_TEMPERATURE,
    top_p: float = DEFAULT_TOP_P,
    top_k: int = DEFAULT_TOP_K,
    enable_thinking: bool = False,
) -> str:
    """
    Generates a single response. `seed` is required -- see design decision
    3 above. Returns the decoded response text only (prompt not included).
    """
    messages = []
    if system_prompt:
        messages.append({"role": "system", "content": system_prompt})
    messages.append({"role": "user", "content": prompt})

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=enable_thinking,
    )
    model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

    torch.manual_seed(seed)
    generated_ids = model.generate(
        **model_inputs,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=temperature,
        top_p=top_p,
        top_k=top_k,
    )
    output_ids = generated_ids[0][len(model_inputs.input_ids[0]):]
    return tokenizer.decode(output_ids, skip_special_tokens=True)


In [ ]:
%%writefile applicant_profile.py
"""
applicant_profile.py

"""

import sys
import warnings

import pandas as pd

# canonical_name -> (human-readable label, unit string or None)
FEATURE_DESCRIPTIONS = {
    "ExternalRiskEstimate": ("consolidated external risk estimate (higher values indicate lower credit risk)", None),
    "MSinceOldestTradeOpen": ("months since the oldest credit trade was opened", "months"),
    "MSinceMostRecentTradeOpen": ("months since the most recent credit trade was opened", "months"),
    "AverageMInFile": ("average number of months trade lines have been on file", "months"),
    "NumSatisfactoryTrades": ("number of satisfactory trades", None),
    "NumTrades60Ever2DerogPubRec": ("number of trades ever 60+ days delinquent or with a derogatory public record", None),
    "NumTrades90Ever2DerogPubRec": ("number of trades ever 90+ days delinquent or with a derogatory public record", None),
    "PercentTradesNeverDelq": ("percentage of trades never delinquent", "%"),
    "MSinceMostRecentDelq": ("months since the most recent delinquency", "months"),
    "MaxDelq2PublicRecLast12M": ("worst delinquency/public-record status in the last 12 months", None),
    "MaxDelqEver": ("worst delinquency/public-record status ever recorded", None),
    "NumTotalTrades": ("total number of trades", None),
    "NumTradesOpeninLast12M": ("number of trades opened in the last 12 months", None),
    "PercentInstallTrades": ("percentage of trades that are installment trades", "%"),
    "MSinceMostRecentInqexcl7days": ("months since the most recent credit inquiry (excluding the last 7 days)", "months"),
    "NumInqLast6M": ("number of credit inquiries in the last 6 months", None),
    "NumInqLast6Mexcl7days": ("number of credit inquiries in the last 6 months, excluding the last 7 days", None),
    "NetFractionRevolvingBurden": ("revolving balance as a percentage of the revolving credit limit", "%"),
    "NetFractionInstallBurden": ("installment balance as a percentage of the original installment loan amount", "%"),
    "NumRevolvingTradesWBalance": ("number of revolving trades carrying a balance", None),
    "NumInstallTradesWBalance": ("number of installment trades carrying a balance", None),
    "NumBank2NatlTradesWHighUtilization": ("number of bank/national trades with high utilisation", None),
    "PercentTradesWBalance": ("percentage of trades carrying a balance", "%"),
}


MAXDELQ_12M_LABELS = {
    0: "a derogatory public record",
    1: "120+ days delinquent",
    2: "90 days delinquent",
    3: "60 days delinquent",
    4: "30 days delinquent",
    5: "unknown delinquency status",
    6: "unknown delinquency status",
    7: "current and never delinquent",
    8: "an other/unclassified status",
    9: "an other/unclassified status",
}
MAXDELQ_EVER_LABELS = {
    2: "a derogatory public record",
    3: "120+ days delinquent",
    4: "90 days delinquent",
    5: "60 days delinquent",
    6: "30 days delinquent",
    7: "unknown delinquency status",
    8: "current and never delinquent",
    9: "an other/unclassified status",
}

PROFILE_SECTION_ORDER = [
    ["ExternalRiskEstimate"],
    ["MSinceOldestTradeOpen", "MSinceMostRecentTradeOpen", "AverageMInFile",
     "NumTotalTrades", "NumTradesOpeninLast12M", "NumSatisfactoryTrades"],
    ["NumTrades60Ever2DerogPubRec", "NumTrades90Ever2DerogPubRec",
     "PercentTradesNeverDelq", "MSinceMostRecentDelq",
     "MaxDelq2PublicRecLast12M", "MaxDelqEver"],
    ["MSinceMostRecentInqexcl7days", "NumInqLast6M", "NumInqLast6Mexcl7days"],
    ["PercentInstallTrades", "NetFractionRevolvingBurden", "NetFractionInstallBurden",
     "NumRevolvingTradesWBalance", "NumInstallTradesWBalance",
     "NumBank2NatlTradesWHighUtilization", "PercentTradesWBalance"],
]

SENTINEL_PHRASING = {
    "no_bureau_record": "no data is on file for this applicant (no bureau record or no investigation)",
    "no_usable_trades_or_inquiries": "this measure is not available (no usable or valid trades or inquiries of this type)",
    "condition_not_met": "this measure is not applicable (the relevant condition was not met, e.g. no inquiries or no delinquencies on record)",
}


def _describe_maxdelq(feature_name: str, value: float) -> str:
    mapping = MAXDELQ_12M_LABELS if feature_name == "MaxDelq2PublicRecLast12M" else MAXDELQ_EVER_LABELS
    label, _ = FEATURE_DESCRIPTIONS[feature_name]
    code = int(value)
    if code in mapping:
        return f"The applicant's {label} is: {mapping[code]}."
    warnings.warn(
        f"{feature_name}: code {code} not in the confirmed mapping. This "
        f"mapping is now the complete official scheme from FICO's data "
        f"dictionary, so an unmapped code here indicates a genuine data "
        f"anomaly (e.g. MaxDelqEver==1, documented as 'No such value'), "
        f"not a documentation gap -- worth investigating this specific row."
    )
    return (
        f"The applicant's {label} is coded {code}, which does not match "
        f"any documented category for this field (this is unexpected and "
        f"worth investigating)."
    )


def _describe_feature(feature_name: str, row: pd.Series, indicator_cols: list[str]) -> str:
    label, unit = FEATURE_DESCRIPTIONS[feature_name]

    fired_indicator = None
    for ind_col in indicator_cols:
        if ind_col.startswith(f"{feature_name}_missing_") and row.get(ind_col, 0) == 1:
            sentinel_key = ind_col[len(f"{feature_name}_missing_"):]
            fired_indicator = sentinel_key
            break

    if fired_indicator is not None:
        phrase = SENTINEL_PHRASING.get(fired_indicator, "this measure is not available")
        return f"For {label}, {phrase}."

    value = row[feature_name]
    if pd.isna(value):
        warnings.warn(
            f"{feature_name} is NaN but no sentinel indicator fired for this "
            f"row -- unexpected given data_pipeline.py's encoding. Falling "
            f"back to a generic missing-data phrase."
        )
        return f"For {label}, no data is available for this applicant."

    if feature_name in ("MaxDelq2PublicRecLast12M", "MaxDelqEver"):
        return _describe_maxdelq(feature_name, value)

    if unit == "%":
        return f"The applicant's {label} is {value:.1f}%."
    if isinstance(value, float) and value.is_integer():
        value = int(value)
    return f"The applicant's {label} is {value}" + (f" {unit}." if unit else ".")


def build_profile_text(row: pd.Series, metadata: dict) -> str:
    indicator_cols = metadata["indicator_cols"]
    sentences = []
    for section in PROFILE_SECTION_ORDER:
        for feature_name in section:
            if feature_name not in FEATURE_DESCRIPTIONS:
                continue
            sentences.append(_describe_feature(feature_name, row, indicator_cols))
    return " ".join(sentences)


def check_coverage(processed_dir):
    """Run against real processed data to see how often the MaxDelq fallback
    fires -- if it never fires, the confirmed mapping was sufficient."""
    import json
    from pathlib import Path

    processed_dir = Path(processed_dir)
    with open(processed_dir / "metadata.json") as f:
        metadata = json.load(f)
    train = pd.read_csv(processed_dir / "train.csv")

    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always")
        for _, row in train.iterrows():
            build_profile_text(row, metadata)
        fallback_warnings = [w for w in caught if "not in the confirmed mapping" in str(w.message)]

    print(f"Checked {len(train)} rows.")
    print(f"MaxDelq fallback fired {len(fallback_warnings)} times.")
    if fallback_warnings:
        print("Unmapped codes encountered -- since the mapping is now FICO's "
              "complete official scheme, these represent genuine anomalies "
              "in the data (e.g. MaxDelqEver==1) rather than a documentation "
              "gap. Worth checking which rows and why.")
    else:
        print("No unmapped codes found. Every value in this data matches "
              "FICO's documented MaxDelq categories.")


if __name__ == "__main__":
    if len(sys.argv) > 1 and sys.argv[1] == "--check-coverage":
        check_coverage(sys.argv[2] if len(sys.argv) > 2 else "data/processed")
    else:
        print("Usage: python applicant_profile.py --check-coverage [processed_dir]")


In [ ]:
%%writefile stock_pipeline.py
"""
stock_pipeline.py

"""

import re

SYSTEM_PROMPT = (
    "You are a financial assistant helping to assess credit risk for loan "
    "applications. You will be given a summary of an applicant's credit "
    "profile. Based on this information, classify the applicant's credit "
    "risk."
)

USER_PROMPT_TEMPLATE = """Here is the applicant's credit profile:

{profile_text}

Based on this information, classify this applicant's credit risk performance as either "Good" or "Bad". Also provide a confidence score from 0 to 100, and a short explanation for your classification.

Respond in exactly this format:
Classification: <Good or Bad>
Confidence: <a number from 0 to 100>
Explanation: <one or two sentences>"""


def build_stock_prompt(profile_text: str) -> tuple[str, str]:
    """Returns (system_prompt, user_prompt)."""
    return SYSTEM_PROMPT, USER_PROMPT_TEMPLATE.format(profile_text=profile_text)


_CLASSIFICATION_RE = re.compile(r"classification:?\**\s*\**\s*(\w+)", re.IGNORECASE)
_CONFIDENCE_RE = re.compile(r"confidence:?\**\s*\**\s*(\d+\.?\d*)\s*%?", re.IGNORECASE)
_EXPLANATION_RE = re.compile(r"explanation:?\**\s*\**\s*(.+)", re.IGNORECASE | re.DOTALL)


def parse_stock_response(raw_text: str) -> dict:
    """
    Best-effort extraction, no retry/correction. Returns a dict with label,
    confidence, explanation (any of which may be None if not found or
    invalid) and parse_success (True only if all three are valid).
    """
    result = {"label": None, "confidence": None, "explanation": None, "parse_success": False}

    label_match = _CLASSIFICATION_RE.search(raw_text)
    if label_match:
        candidate = label_match.group(1).strip().lower()
        if candidate in ("good", "bad"):
            result["label"] = candidate.capitalize()

    conf_match = _CONFIDENCE_RE.search(raw_text)
    if conf_match:
        try:
            value = float(conf_match.group(1))
            if 0 <= value <= 100:
                result["confidence"] = value
        except ValueError:
            pass

    exp_match = _EXPLANATION_RE.search(raw_text)
    if exp_match:
        explanation = exp_match.group(1).strip()
        if explanation:
            result["explanation"] = explanation

    result["parse_success"] = all([
        result["label"] is not None,
        result["confidence"] is not None,
        result["explanation"] is not None,
    ])
    return result


In [ ]:
%%writefile run_llm_pipeline.py
"""
run_llm_pipeline.py

"""

import argparse
import json
import time
from pathlib import Path

import pandas as pd

from applicant_profile import build_profile_text


def get_pipeline_functions(name: str):
    if name == "stock":
        from stock_pipeline import build_stock_prompt, parse_stock_response
        return build_stock_prompt, parse_stock_response
    if name == "structured":
        raise NotImplementedError(
            "structured pipeline not built yet -- once structured_pipeline.py "
            "exists with build_structured_prompt/parse_structured_response, "
            "add it here the same way 'stock' is wired above."
        )
    raise ValueError(f"Unknown pipeline: {name}")


def load_completed(output_path: Path) -> set:
    completed = set()
    if output_path.exists():
        with open(output_path) as f:
            for line in f:
                line = line.strip()
                if line:
                    rec = json.loads(line)
                    completed.add((rec["case_id"], rec["seed"]))
    return completed


def run_pipeline(
    subset: pd.DataFrame,
    metadata: dict,
    pipeline_name: str,
    build_prompt_fn,
    parse_fn,
    generate_fn,
    output_path: Path,
    n_repeats: int,
    max_new_tokens: int,
    progress_every: int = 10,
) -> int:
    """
    generate_fn(user_prompt: str, seed: int, system_prompt: str,
    max_new_tokens: int) -> str. Returns the number of generations run
    (excludes ones skipped because already completed).
    """
    completed = load_completed(output_path)
    todo = [
        (idx, row, seed)
        for idx, row in subset.iterrows()
        for seed in range(n_repeats)
        if (row["case_id"], seed) not in completed
    ]

    output_path.parent.mkdir(parents=True, exist_ok=True)
    n_run = 0
    with open(output_path, "a") as out_f:
        for i, (idx, row, seed) in enumerate(todo):
            profile_text = build_profile_text(row, metadata)
            system_prompt, user_prompt = build_prompt_fn(profile_text)

            start = time.time()
            raw_response = generate_fn(user_prompt, seed, system_prompt, max_new_tokens)
            elapsed = time.time() - start

            parsed = parse_fn(raw_response)

            record = {
                "case_id": row["case_id"],
                "pipeline": pipeline_name,
                "seed": seed,
                "true_label": row[metadata["target_col"]],
                "raw_response": raw_response,
                "elapsed_seconds": elapsed,
                **parsed,
            }
            out_f.write(json.dumps(record) + "\n")
            out_f.flush()
            n_run += 1

            if n_run % progress_every == 0:
                print(
                    f"[{n_run}/{len(todo)}] case={row['case_id']} seed={seed} "
                    f"parse_success={parsed['parse_success']} elapsed={elapsed:.1f}s"
                )

    return n_run


def main():
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--pipeline", choices=["stock", "structured"], required=True)
    parser.add_argument("--subset-path", type=Path, required=True)
    parser.add_argument("--metadata-path", type=Path, required=True)
    parser.add_argument("--output-path", type=Path, required=True)
    parser.add_argument("--n-repeats", type=int, default=5)
    parser.add_argument("--max-new-tokens", type=int, default=400)
    args = parser.parse_args()

    build_prompt_fn, parse_fn = get_pipeline_functions(args.pipeline)

    subset = pd.read_csv(args.subset_path)
    with open(args.metadata_path) as f:
        metadata = json.load(f)

    print("Loading model...")
    from qwen3_inference import load_model_and_tokenizer, generate
    model, tokenizer = load_model_and_tokenizer(quantize_4bit=True)
    print("Model loaded.")

    def generate_fn(user_prompt, seed, system_prompt, max_new_tokens):
        return generate(
            model, tokenizer, user_prompt, seed=seed,
            system_prompt=system_prompt, max_new_tokens=max_new_tokens,
        )

    total_needed = len(subset) * args.n_repeats
    already_done = len(load_completed(args.output_path))
    print(f"{already_done}/{total_needed} (case_id, seed) pairs already completed.")

    n_run = run_pipeline(
        subset, metadata, args.pipeline, build_prompt_fn, parse_fn, generate_fn,
        args.output_path, args.n_repeats, args.max_new_tokens,
    )
    print(f"Done. Ran {n_run} new generations. Results in {args.output_path}")


if __name__ == "__main__":
    main()


## 4. Point at your uploaded data on Drive

Confirms `metadata.json` and `llm_eval_subset.csv` are actually where this
notebook expects them, and previews the subset

In [ ]:
import pandas as pd, json, os

METADATA_PATH = f"{DRIVE_PROJECT_DIR}/data/metadata.json"
SUBSET_PATH = f"{DRIVE_PROJECT_DIR}/data/llm_eval_subset.csv"
OUTPUT_PATH = f"{DRIVE_PROJECT_DIR}/results/stock_results.jsonl"

for p in (METADATA_PATH, SUBSET_PATH):
    if not os.path.exists(p):
        raise FileNotFoundError(
            f"{p} not found -- upload it to Drive first (see intro cell above)."
        )

with open(METADATA_PATH) as f:
    metadata = json.load(f)
subset = pd.read_csv(SUBSET_PATH)

print(f"Subset: {len(subset)} cases")
print(f"Class balance:\n{subset[metadata['target_col']].value_counts()}")
print(f"Output will be written to: {OUTPUT_PATH}")

## 5. Run the stock pipeline

In [ ]:
!python run_llm_pipeline.py \
    --pipeline stock \
    --subset-path "{SUBSET_PATH}" \
    --metadata-path "{METADATA_PATH}" \
    --output-path "{OUTPUT_PATH}" \
    --n-repeats 5 \
    --max-new-tokens 400

## 6. Check the results

In [ ]:
results = pd.read_json(OUTPUT_PATH, lines=True)

print(f"Total records: {len(results)}")
print(f"Expected: {len(subset) * 5} ({len(subset)} cases x 5 repeats)")
print(f"Unique cases covered: {results['case_id'].nunique()} / {len(subset)}")
print(f"Parse success rate: {results['parse_success'].mean():.1%}")
print(f"Mean generation time: {results['elapsed_seconds'].mean():.1f}s")

incomplete = len(subset) * 5 - len(results)
if incomplete > 0:
    print(f"\n{incomplete} generations still remaining -- re-run Cell 5 to continue.")
else:
    print("\nAll generations complete.")